In [0]:
notes_test = spark.read.table("...")
notes_train = spark.read.table("...")

In [0]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

import os, random, math, time, gc
from dataclasses import dataclass, asdict
from typing import List, Dict, Any

import pandas as pd
import numpy as np
from scipy import stats

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AdamW,
    get_linear_schedule_with_warmup,
    logging as hf_logging,
)
from sklearn.metrics import precision_recall_fscore_support

hf_logging.set_verbosity_error()        
spark = SparkSession.builder.getOrCreate()


MODEL_NAME          = "emilyalsentzer/Bio_ClinicalBERT"        
DEVICE              = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE          = 8                                  
N_EPOCHS            = 4                                  
LEARNING_RATE       = 2e-5
WARMUP_FRAC         = 0.1                               
TRAIN_SIZES         = [10, 25, 50, 100, 150, 200]
N_REPEATS           = 20                                 


@dataclass
class MetricsRow:
    task: str      
    train_size: int
    label: str      n    metric: str     
    mean: float
    ci_lower: float
    ci_upper: float


# Build input strings so model “sees” patient context

def build_input_string(row: pd.Series) -> str:
    patient_text = row["patient_message_text"]
    provider_text = row["provider_message_text"]

    patient_labels = (
        f"EMOTION={row['statement_of_emotion']} "
        f"NEG_VAL={row['valence_negative']} "
        f"POS_VAL={row['valence_positive']} "
        f"PROGRESS={row['statement_of_progress']} "
        f"CHALLENGE={row['statement_of_challenge']}"
    )
    return f"PATIENT: {patient_text} [SEP] PROVIDER: {provider_text} [SEP] PATIENT_LABELS: {patient_labels}"


# Dataset wrapper

class ProviderDataset(Dataset):
    def __init__(
        self,
        pdf: pd.DataFrame,
        tokenizer: AutoTokenizer,
        max_len: int,
        task: str,
    ):
        texts = pdf.apply(build_input_string, axis=1).tolist()

        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=max_len,
        )

        if task == "multiclass":
            # Collect exactly one label per row (columns 10-15)
            label_cols = [
                "forwarded",
                "denial",
                "implicit_recognition",
                "acknowledgement",
                "confirmation",
                "shared_feeling",
            ]
            self.labels = pdf[label_cols].values.argmax(axis=1)
        else:  # binary
            self.labels = pdf["empathy"].astype(int).values

        self.task = task

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        if self.task == "multiclass":
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        else:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


# 5.  Training & evaluation 

def train_one_epoch(model, loader, optimiser, scheduler):
    model.train()
    total_loss = 0.0
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss / 1  
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimiser.step()
        scheduler.step()
        optimiser.zero_grad()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model, loader, task, label_names) -> Dict[str, Any]:
    model.eval()
    preds, targets = [], []

    for batch in loader:
        labels = batch.pop("labels")
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = model(**batch)
        logits = outputs.logits.detach().cpu()

        if task == "multiclass":
            preds.extend(logits.argmax(dim=1).tolist())
            targets.extend(labels.tolist())
        else:
            preds.extend(logits.argmax(dim=1).tolist())
            targets.extend(labels.tolist())

    preds = np.array(preds)
    targets = np.array(targets)

    # micro, macro, per-label
    metrics = {}
    p, r, f, _ = precision_recall_fscore_support(targets, preds, average="micro", zero_division=0)
    metrics["micro"] = (p, r, f)
    p, r, f, _ = precision_recall_fscore_support(targets, preds, average="macro", zero_division=0)
    metrics["macro"] = (p, r, f)
    p, r, f, _ = precision_recall_fscore_support(targets, preds, average=None, zero_division=0, labels=range(len(label_names)))
    for idx, name in enumerate(label_names):
        metrics[name] = (p[idx], r[idx], f[idx])
    return metrics

def ci95(values: List[float]) -> (float, float):
    if len(values) == 1:          # training size = 200
        return values[0], values[0]
    mean = np.mean(values)
    h = stats.t.ppf(0.975, len(values) - 1) * stats.sem(values)
    return mean - h, mean + h


# Main experiment loop

def run_experiments(model_name: str):

    train_pdf_full = notes_train.toPandas()
    test_pdf       = notes_test.toPandas()

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    def get_max_len(tok, fallback=512, threshold=4096):
        return tok.model_max_length if tok.model_max_length < threshold else fallback
    
    max_len = get_max_len(tokenizer)

    results: List[MetricsRow] = []

    #   Run both tasks

    for task, label_names in [
        ("multiclass", ["forwarded", "denial", "implicit_recognition", "acknowledgement", "confirmation", "shared_feeling"]),
        ("binary",     ["negative", "positive"] if model_name != "emilyalsentzer/Bio_ClinicalBERT" else ["no_empathy", "empathy"]),  
    ]:

        test_ds = ProviderDataset(test_pdf, tokenizer, max_len, task)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

        for train_size in TRAIN_SIZES:
            reps = 1 if train_size == len(train_pdf_full) else N_REPEATS

            metrics_store: Dict[str, Dict[str, List[float]]] = {
                lbl: {"precision": [], "recall": [], "f1": []}
                for lbl in (["micro", "macro"] + label_names)
            }

            for rep in range(reps):
                train_pdf = train_pdf_full.sample(
                    n=train_size,
                    replace=False,
                    random_state=rep,     
                )
                train_ds = ProviderDataset(train_pdf, tokenizer, max_len, task)
                train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

                
                #   Model/optimiser
                
                num_labels = 6 if task == "multiclass" else 2
                problem_type = "single_label_classification"
                model = AutoModelForSequenceClassification.from_pretrained(
                    model_name,
                    num_labels=num_labels,
                    problem_type=problem_type,
                ).to(DEVICE)

                total_steps = len(train_loader) * N_EPOCHS
                optimiser   = AdamW(model.parameters(), lr=LEARNING_RATE)
                scheduler   = get_linear_schedule_with_warmup(
                    optimiser,
                    num_warmup_steps=int(total_steps * WARMUP_FRAC),
                    num_training_steps=total_steps,
                )

                #   Training loop

                for epoch in range(N_EPOCHS):
                    _ = train_one_epoch(model, train_loader, optimiser, scheduler)

                #   Evaluation on test set

                run_metrics = evaluate(model, test_loader, task, label_names)

                for lbl, (p, r, f) in run_metrics.items():
                    metrics_store[lbl]["precision"].append(p)
                    metrics_store[lbl]["recall"].append(r)
                    metrics_store[lbl]["f1"].append(f)

                del model, optimiser, scheduler, train_loader, train_ds
                torch.cuda.empty_cache()
                gc.collect()


            for lbl, metric_dict in metrics_store.items():
                for metric_name, values in metric_dict.items():
                    mean = float(np.mean(values))
                    ci_lo, ci_hi = ci95(values)
                    results.append(
                        MetricsRow(
                            task=task,
                            train_size=train_size,
                            label=lbl,
                            metric=metric_name,
                            mean=mean,
                            ci_lower=ci_lo,
                            ci_upper=ci_hi,
                        )
                    )


    results_pdf = pd.DataFrame([asdict(r) for r in results])
    results_sdf = spark.createDataFrame(results_pdf)
    return results_sdf.filter(F.col("task") == "multiclass"), results_sdf.filter(F.col("task") == "binary")


start = time.time()
multiclass_results_sdf, binary_results_sdf = run_experiments(MODEL_NAME)
print(f"Finished in {time.time() - start:0.1f}s")

In [0]:
display(multiclass_results_sdf.orderBy('train_size', 'label'))


In [0]:
display(binary_results_sdf.orderBy('train_size', 'label'))